[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/04_wordcloud.ipynb)


# 4-4차시 텍스트분석 실습 ② — 단어빈도수 워드클라우드 시각화

①에서 센 단어 빈도수를 **그림**으로 표현합니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 🔴 **맥 사용자 필독**
> 영상 코드의 `font_path="malgun.ttf"` 는 **윈도우 전용 글꼴**이라
> 맥에서 그대로 실행하면 오류가 납니다.
> 이 노트북은 **나눔고딕을 설치해** 운영체제와 상관없이 똑같은 결과가 나오게 했습니다.


## 코드 설계

| 단계 | 하는 일 |
|---|---|
| **1단계** | 텍스트 가져오기 |
| **2단계** | 텍스트 전처리 — 불용어 제거, 형태소 분석 |
| **3단계** | 단어빈도수 계산 |
| **4단계** | 워드클라우드 생성 및 출력 |

활용 라이브러리: **konlpy, wordcloud, matplotlib**


---
## 준비 — 라이브러리 설치

영상의 `pip install konlpy wordcloud matplotlib` 에 해당합니다.
여기에 **한글 글꼴(나눔고딕)** 설치를 더했습니다. (1~2분)


In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!apt-get install -y -qq fonts-nanum > /dev/null   # <- 한글 글꼴 (맑은 고딕 대신)
!pip install -q konlpy wordcloud matplotlib
!fc-cache -f > /dev/null

import glob, os

jvm = sorted(glob.glob('/usr/lib/jvm/java-*-openjdk-amd64'))
os.environ['JAVA_HOME'] = jvm[-1]

# 설치된 한글 글꼴 경로를 찾아 둡니다
FONT_PATH = glob.glob('/usr/share/fonts/**/NanumGothic.ttf', recursive=True)[0]
print('한글 글꼴:', FONT_PATH)
print('설치 완료!')


## 준비 — 실습 데이터 내려받기


In [ ]:
# 실습 데이터를 강의 깃허브 저장소에서 곧바로 내려받습니다.
# 내 컴퓨터의 파일 경로를 적을 필요가 없습니다.

GITHUB = "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc"

FILES = [
    "data/word-analysis/kim-sangheon-sangso.txt",
]

import os
import urllib.request

for path in FILES:
    name = path.split('/')[-1]
    try:
        urllib.request.urlretrieve(f'{GITHUB}/{path}', name)
    except Exception as e:
        raise SystemExit(
            f'내려받기에 실패했습니다: {path}\n'
            f'  · 인터넷 연결을 확인해 주세요.\n'
            f'  · 그래도 안 되면 강의 게시판에 문의해 주세요.\n'
            f'  (원인: {e})'
        )
    print(f'내려받음: {name}  ({os.path.getsize(name):,} 바이트)')

print()
print('준비 완료! 아래 칸부터 차례로 실행하세요.')

# (선택) 다른 텍스트로 해보고 싶다면 아래 두 줄의 # 을 지우고 실행하세요.
# from google.colab import files
# files.upload()


---
## 1~3단계 — 텍스트 가져오기 → 전처리 → 빈도수 계산

① 실습과 똑같은 과정입니다. 한 칸에 모았습니다.


In [ ]:
import re
from collections import Counter
from konlpy.tag import Okt

# 1단계: 텍스트 가져오기
file_path = 'kim-sangheon-sangso.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

# 2단계: 텍스트 전처리
text = re.sub(r'[^가-힣\s]', '', text)   # 한글 이외의 문자 제거
okt = Okt()                              # 형태소 분석기 사용 (명사 추출)
tokens = okt.nouns(text)

# 3단계: 단어 빈도수 계산
word_counts = Counter(tokens)

print(f'단어 {len(word_counts):,}종류')
print(word_counts.most_common(10))


> ✅ 맨 앞이 `('것', 157)` 로 시작하면 제대로 된 것입니다.


---
## 4단계 — 워드클라우드 생성 및 출력

`font_path` 에 위에서 찾아 둔 나눔고딕 경로를 넣는 것이 **한글이 깨지지 않는 핵심**입니다.


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 워드클라우드 생성
wordcloud = WordCloud(
    font_path=FONT_PATH,      # <- 윈도우 전용 'malgun.ttf' 대신 이걸 씁니다
    background_color='white',
    width=800,
    height=600
).generate_from_frequencies(word_counts)

# 워드클라우드 출력
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # 축 제거
plt.show()


> ✅ **`것` 이 가장 크게** 나오고 `수`, `그`, `때`, `말` 이 뒤따르면 영상과 같은 결과입니다.


### 그림 파일로 내려받기


In [ ]:
wordcloud.to_file('wordcloud.png')

from google.colab import files
files.download('wordcloud.png')


---
## 더 해보기 1 — 불용어 걸러내고 다시 그리기

> 여기부터는 **교안에 없는 내용**입니다. 결과가 영상과 달라집니다.

`것`, `수`, `그` 처럼 **어느 글에나 나오는 말**을 빼면,
이 상소문이 실제로 무엇을 말하는 글인지 훨씬 잘 드러납니다.


In [ ]:
stopwords = ['것', '저', '그', '이', '수', '때', '등', '바', '더']

filtered = [w for w in tokens if w not in stopwords and len(w) > 1]
filtered_counts = Counter(filtered)

print(filtered_counts.most_common(10))

wordcloud2 = WordCloud(
    font_path=FONT_PATH,
    background_color='white',
    width=800,
    height=600
).generate_from_frequencies(filtered_counts)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud2, interpolation='bilinear')
plt.axis('off')
plt.show()


## 더 해보기 2 — 모양 바꾸기

아래 값을 바꿔 가며 여러 번 실행해 보세요.

| 옵션 | 뜻 | 예시 |
|---|---|---|
| `background_color` | 배경색 | `'white'`, `'black'` |
| `max_words` | 보여줄 단어 수 | `200`(기본), `80` |
| `colormap` | 색조합 | `'viridis'`(기본), `'Blues'`, `'autumn'` |
| `width` / `height` | 크기 | `1600` / `1200` |


In [ ]:
wordcloud3 = WordCloud(
    font_path=FONT_PATH,
    background_color='black',
    colormap='autumn',
    max_words=80,
    width=1600,
    height=1200
).generate_from_frequencies(filtered_counts)

plt.figure(figsize=(12, 9))
plt.imshow(wordcloud3, interpolation='bilinear')
plt.axis('off')
plt.show()


---
### 정리

1. 단어빈도수 분석은 **텍스트 전처리 → 빈도수 분석** 의 순서로 코드를 작성한다.
2. 단어빈도수 분석에 어울리는 **시각화**를 할 수 있다.
